In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

In [12]:
RAW = Path("../data/raw")

In [13]:
PROCESSED = Path("../data/processed")

In [14]:
for file in RAW.glob("*.csv"):
    print(file.name)

Bookings_2023.csv
Bookings_2024.csv
Bookings_2025.csv


In [15]:
df_2023 = pd.read_csv(RAW / "Bookings_2023.csv")
df_2024 = pd.read_csv(RAW / "Bookings_2024.csv")
df_2025 = pd.read_csv(RAW / "Bookings_2025.csv")

In [18]:
df.shape


(13927, 12)

In [17]:
df = pd.concat([df_2023, df_2024, df_2025], ignore_index=True)

In [9]:
# 1. Remove duplicate bookings
df = df.drop_duplicates(subset="BookingID", keep="first")

# 2. Fix dates
df["BookingDate"] = pd.to_datetime(df["BookingDate"], errors="coerce")
df["TravelStartDate"] = pd.to_datetime(df["TravelStartDate"], errors="coerce")
df["BookingYear"] = df["BookingDate"].dt.year
df["BookingMonth"] = df["BookingDate"].dt.month

# 3. Clean the Status text
df["Status"] = df["Status"].str.strip().str.lower().replace({"canceled": "cancelled"})

# 4. Fill missing values
df["DiscountPctClean"] = df["DiscountPct"].fillna(0)
df["AgentClean"] = df["AgentID"].fillna("Unknown")

In [13]:
# numpy practice
arr = np.array([1,2,3,4], dtype=np.float64)
print(arr)
print(type(arr))

[1. 2. 3. 4.]
<class 'numpy.ndarray'>


In [12]:
print(arr.ndim)

1


In [5]:
matrix = np.array([[1,3,5,6], [4,6,7,8]])
matrix.shape
matrix.size

8

In [10]:
zeros_arr =np.zeros(3)
print(zeros_arr)

[0. 0. 0.]


In [23]:
prices = df_2024["UnitPriceUSD"].to_numpy()
print(prices.dtype)

int64


In [25]:
# indexing
print(prices[:10])
print(prices[-3:])

[  940  7485   565  2470  4830 11835   400  1510  4095   505]
[ 595 2860 6580]


In [31]:
ar_adults =df_2024["Adults"].to_numpy()
print(ar_adults[:10])

[2 1 5 2 2 2 8 2 2 2]


In [27]:
df_2024.head()


,BookingID,BookingDate,TravelStartDate,CustomerID,PackageID,AgentID,Adults,Children,UnitPriceUSD,DiscountPct,Status,PaymentMethod
0,BK-2024-00001,2024-01-01,2024-01-17,C00037,PK05,AG10,2,1,940,0.05,Completed,Card
1,BK-2024-00002,2024-01-01,2024-08-08,C00040,PK13,AG03,1,0,7485,0.00,Completed,Card
2,BK-2024-00003,2024-01-01,2024-02-14,C00381,PK03,AG12,5,0,565,0.00,Completed,PayPal
3,BK-2024-00004,2024-01-01,2024-03-22,C00500,PK07,AG00,2,1,2470,0.00,Completed,Card
4,BK-2024-00005,2024-01-01,2024-05-25,C00880,PK16,AG08,2,3,4830,0.05,Completed,Card


In [36]:
# creating a two dimensional array
adult_pricing =df_2024[["Adults", "UnitPriceUSD"]].to_numpy()
print(adult_pricing)

[[   2  940]
 [   1 7485]
 [   5  565]
 ...
 [  12  595]
 [   2 2860]
 [  10 6580]]


In [39]:
combined_mask = (adult_pricing[:, 0] > 2) & (adult_pricing[:, 1] < 50)
print(combined_mask)

[False False False ... False False False]


In [41]:
total_price = np.sum(adult_pricing[:, 1])
print(total_price)

14385610


In [42]:
discounts = df_2024["DiscountPct"].to_numpy()
print("Missing Counts", np.isnan(discounts).sum())

Missing Counts 133


In [44]:
df["Travellers"] = df["Adults"] + df["Children"]
df.head()

,BookingID,BookingDate,TravelStartDate,CustomerID,PackageID,AgentID,Adults,Children,UnitPriceUSD,DiscountPct,Status,PaymentMethod,Travellers
0,BK-2023-00001,2023-01-01,2023-02-20,C01696,PK06,AG10,2,0,1250,0.00,Completed,Bank Transfer,2
1,BK-2023-00002,2023-01-01,2023-07-07,C01980,PK09,AG02,1,0,2990,0.00,Completed,Card,1
2,BK-2023-00003,2023-01-01,2023-08-27,C01992,PK15,AG08,2,3,4485,0.00,Completed,PayPal,5
3,BK-2023-00004,2023-01-01,2023-03-27,C02537,PK06,NaN,1,0,1250,0.00,Completed,Card,1
4,BK-2023-00005,2023-01-01,2023-02-08,C03846,PK01,NaN,1,0,420,0.15,Completed,Bank Transfer,1


In [45]:
df["GrossValue"] = df["UnitPriceUSD"]

In [46]:
df["DiscountPctClean"] = df["DiscountPct"].fillna(0)

In [47]:
df["DiscountAmount"] = df["GrossValue"] * df["DiscountPctClean"]

In [49]:
#Net booking value
df["NetBookingValue"] = df["GrossValue"] - df["DiscountAmount"]
df.head()

,BookingID,BookingDate,TravelStartDate,CustomerID,PackageID,AgentID,Adults,Children,UnitPriceUSD,DiscountPct,Status,PaymentMethod,Travellers,GrossValue,DiscountPctClean,DiscountAmount,NetBookingValue
0,BK-2023-00001,2023-01-01,2023-02-20,C01696,PK06,AG10,2,0,1250,0.00,Completed,Bank Transfer,2,1250,0.00,0.0,1250.0
1,BK-2023-00002,2023-01-01,2023-07-07,C01980,PK09,AG02,1,0,2990,0.00,Completed,Card,1,2990,0.00,0.0,2990.0
2,BK-2023-00003,2023-01-01,2023-08-27,C01992,PK15,AG08,2,3,4485,0.00,Completed,PayPal,5,4485,0.00,0.0,4485.0
3,BK-2023-00004,2023-01-01,2023-03-27,C02537,PK06,NaN,1,0,1250,0.00,Completed,Card,1,1250,0.00,0.0,1250.0
4,BK-2023-00005,2023-01-01,2023-02-08,C03846,PK01,NaN,1,0,420,0.15,Completed,Bank Transfer,1,420,0.15,63.0,357.0
